In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import time
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from src.preprocessing import load_dataset, preprocess
from src.classifiers import get_classifier_config, CLASSIFIER_CONFIGS
from src.csa_optimizer import optimize_classifier

sns.set_style("whitegrid")
RANDOM_STATE = 42
print("All imports successful")

All imports successful


In [2]:
# Dataset configurations
DATASETS = {
    "UNR-IDD": {
        "path": "../data/UNR-IDD.csv",
        "label_col": "Label",
        "drop_cols": [
            "Binary Label",
            "Packets Rx Dropped", "Packets Tx Dropped",
            "Packets Rx Errors", "Packets Tx Errors",
            "Table ID", "Max Size"
        ],
        "type": "single_file"
    },
    "CICIDS2017": {
        "path": "../data/CICIDS2017/",
        "label_col": " Label",
        "drop_cols": [],
        "type": "multi_file"
    },
    "UNSW-NB15": {
        "path": "../data/UNSW-NB15/",
        "label_col": "label",
        "drop_cols": [],
        "type": "multi_file"
    }
}

CLASSIFIERS = ["KNN", "LR", "RF", "DT", "AB", "XGB", "LGBM"]

print("Dataset configurations loaded")
print(f"Classifiers to evaluate: {CLASSIFIERS}")

Dataset configurations loaded
Classifiers to evaluate: ['KNN', 'LR', 'RF', 'DT', 'AB', 'XGB', 'LGBM']


In [3]:
def load_and_merge(dataset_config):
    """Load single or multi-file dataset and return a clean DataFrame."""
    if dataset_config["type"] == "single_file":
        df = pd.read_csv(dataset_config["path"])
        print(f"Loaded: {df.shape}")
        return df
    
    elif dataset_config["type"] == "multi_file":
        dfs = []
        path = dataset_config["path"]
        for fname in sorted(os.listdir(path)):
            if fname.endswith(".csv"):
                fpath = os.path.join(path, fname)
                try:
                    tmp = pd.read_csv(fpath, encoding="utf-8", low_memory=False)
                    dfs.append(tmp)
                    print(f"  Loaded {fname}: {tmp.shape}")
                except Exception as e:
                    print(f"  Skipped {fname}: {e}")
        df = pd.concat(dfs, ignore_index=True)
        print(f"Combined shape: {df.shape}")
        return df

# Test loading UNR-IDD first to confirm pipeline works
print("Loading UNR-IDD...")
df_unr = load_and_merge(DATASETS["UNR-IDD"])
print(f"\nLabel column sample: {df_unr['Label'].value_counts().to_dict()}")

Loading UNR-IDD...
Loaded: (37411, 34)

Label column sample: {'PortScan': 9500, 'TCP-SYN': 9081, 'Blackhole': 8420, 'Diversion': 5615, 'Normal': 3773, 'Overflow': 1022}


In [4]:
print("Loading CICIDS2017...")
df_cic = load_and_merge(DATASETS["CICIDS2017"])
print(f"\nLabel column sample:")
print(df_cic[" Label"].value_counts())

Loading CICIDS2017...
  Loaded Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: (225745, 79)
  Loaded Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: (286467, 79)
  Loaded Friday-WorkingHours-Morning.pcap_ISCX.csv: (191033, 79)
  Loaded Monday-WorkingHours.pcap_ISCX.csv: (529918, 79)
  Loaded Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: (288602, 79)
  Loaded Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: (170366, 79)
  Loaded Tuesday-WorkingHours.pcap_ISCX.csv: (445909, 79)
  Loaded Wednesday-workingHours.pcap_ISCX.csv: (692703, 79)
Combined shape: (2830743, 79)

Label column sample:
 Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                         

In [5]:
print("Loading UNSW-NB15...")
df_unsw = load_and_merge(DATASETS["UNSW-NB15"])
print(f"\nColumns: {df_unsw.columns.tolist()}")
print(f"\nLast few columns: {df_unsw.columns[-5:].tolist()}")

Loading UNSW-NB15...
  Loaded NUSW-NB15_GT.csv: (174347, 12)
  Skipped NUSW-NB15_features.csv: 'utf-8' codec can't decode byte 0x92 in position 72: invalid start byte
  Loaded UNSW-NB15_1.csv: (700000, 49)
  Loaded UNSW-NB15_2.csv: (700000, 49)
  Loaded UNSW-NB15_3.csv: (700000, 49)
  Loaded UNSW-NB15_4.csv: (440043, 49)
  Loaded UNSW-NB15_LIST_EVENTS.csv: (208, 3)
Combined shape: (2714598, 152)

Columns: ['Start time', 'Last time', 'Attack category', 'Attack subcategory', 'Protocol', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Attack Name', 'Attack Reference', '.', '59.166.0.0', '1390', '149.171.126.6', '53', 'udp', 'CON', '0.001055', '132', '164', '31', '29', '0', '0.1', 'dns', '500473.9375', '621800.9375', '2', '2.1', '0.2', '0.3', '0.4', '0.5', '66', '82', '0.6', '0.7', '0.8', '0.9', '1421927414', '1421927414.1', '0.017', '0.013', '0.10', '0.11', '0.12', '0.13', '0.14', '0.15', '0.16', '0.17', '3', '7', '1', '3.1', '1.1', '1.2', '1.3', 'Unnamed: 47', '0.18', 

In [6]:
# Fix UNSW-NB15 loading — only use the 4 main data files
def load_unsw(path):
    dfs = []
    for i in range(1, 5):
        fpath = os.path.join(path, f"UNSW-NB15_{i}.csv")
        tmp = pd.read_csv(fpath, header=None, low_memory=False)
        print(f"  Loaded UNSW-NB15_{i}.csv: {tmp.shape}")
        dfs.append(tmp)
    df = pd.concat(dfs, ignore_index=True)
    print(f"Combined shape: {df.shape}")
    return df

print("Loading UNSW-NB15 (main files only)...")
df_unsw = load_unsw(DATASETS["UNSW-NB15"]["path"])
print(f"\nFirst few columns: {df_unsw.columns.tolist()[:10]}")
print(f"\nLast column (label): {df_unsw.iloc[:, -1].value_counts()}")

Loading UNSW-NB15 (main files only)...
  Loaded UNSW-NB15_1.csv: (700001, 49)
  Loaded UNSW-NB15_2.csv: (700001, 49)
  Loaded UNSW-NB15_3.csv: (700001, 49)
  Loaded UNSW-NB15_4.csv: (440044, 49)
Combined shape: (2540047, 49)

First few columns: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

Last column (label): 48
0    2218764
1     321283
Name: count, dtype: int64


In [7]:
# Check the attack category column (column 47)
print("Column 47 (attack category):")
print(df_unsw.iloc[:, 47].value_counts())

print("\nColumn 48 (binary label):")
print(df_unsw.iloc[:, 48].value_counts())

# Also check what the feature names file says
features_path = os.path.join(DATASETS["UNSW-NB15"]["path"], "NUSW-NB15_features.csv")
try:
    features = pd.read_csv(features_path, encoding="latin-1")
    print("\nFeature names:")
    print(features[["No.", "Name"]].to_string())
except Exception as e:
    print(f"Could not load features file: {e}")

Column 47 (attack category):
47
Generic             215481
Exploits             44525
 Fuzzers             19195
DoS                  16353
 Reconnaissance      12228
 Fuzzers              5051
Analysis              2677
Backdoor              1795
Reconnaissance        1759
 Shellcode            1288
Backdoors              534
Shellcode              223
Worms                  174
Name: count, dtype: int64

Column 48 (binary label):
48
0    2218764
1     321283
Name: count, dtype: int64

Feature names:
    No.              Name
0     1             srcip
1     2             sport
2     3             dstip
3     4            dsport
4     5             proto
5     6             state
6     7               dur
7     8            sbytes
8     9            dbytes
9    10              sttl
10   11              dttl
11   12             sloss
12   13             dloss
13   14           service
14   15             Sload
15   16             Dload
16   17             Spkts
17   18             Dpkts

In [8]:
# Assign proper column names to UNSW-NB15
col_names = [
    "srcip", "sport", "dstip", "dsport", "proto", "state", "dur",
    "sbytes", "dbytes", "sttl", "dttl", "sloss", "dloss", "service",
    "Sload", "Dload", "Spkts", "Dpkts", "swin", "dwin", "stcpb",
    "dtcpb", "smeansz", "dmeansz", "trans_depth", "res_bdy_len",
    "Sjit", "Djit", "Stime", "Ltime", "Sintpkt", "Dintpkt", "tcprtt",
    "synack", "ackdat", "is_sm_ips_ports", "ct_state_ttl",
    "ct_flw_http_mthd", "is_ftp_login", "ct_ftp_cmd", "ct_srv_src",
    "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm", "ct_src_dport_ltm",
    "ct_dst_sport_ltm", "ct_dst_src_ltm", "attack_cat", "label"
]

df_unsw.columns = col_names

# Clean attack_cat — strip whitespace, fill missing with Normal
df_unsw["attack_cat"] = df_unsw["attack_cat"].str.strip()
df_unsw["attack_cat"] = df_unsw["attack_cat"].fillna("Normal")
df_unsw.loc[df_unsw["attack_cat"] == "", "attack_cat"] = "Normal"

# Also fix Backdoors -> Backdoor inconsistency
df_unsw["attack_cat"] = df_unsw["attack_cat"].replace({
    "Backdoors": "Backdoor",
    "Fuzzers": "Fuzzer",
})

print("Class distribution after cleaning:")
print(df_unsw["attack_cat"].value_counts())
print(f"\nTotal samples: {len(df_unsw)}")

Class distribution after cleaning:
attack_cat
Normal            2218764
Generic            215481
Exploits            44525
Fuzzer              24246
DoS                 16353
Reconnaissance      13987
Analysis             2677
Backdoor             2329
Shellcode            1511
Worms                 174
Name: count, dtype: int64

Total samples: 2540047


In [9]:
def preprocess_dataset(df, label_col, drop_cols=None, test_size=0.2, random_state=42):
    """
    Unified preprocessing pipeline for all three datasets.
    Prevents data leakage by fitting only on training data.
    """
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler, LabelEncoder

    # Drop duplicates
    df = df.drop_duplicates()
    print(f"After dedup: {df.shape}")

    # Drop specified columns
    cols_to_drop = [label_col] + (drop_cols or [])
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]
    X = df.drop(columns=cols_to_drop).copy()
    y = df[label_col].copy()

    # Drop zero-variance columns
    zero_var = [c for c in X.columns if X[c].nunique() <= 1]
    if zero_var:
        print(f"Dropping zero-variance columns: {zero_var}")
        X = X.drop(columns=zero_var)

    # Drop columns with too many missing values over 50 percent
    missing_pct = X.isnull().mean()
    high_missing = missing_pct[missing_pct > 0.5].index.tolist()
    if high_missing:
        print(f"Dropping high-missing columns: {high_missing}")
        X = X.drop(columns=high_missing)

    # Encode categorical and bool columns
    for col in X.select_dtypes(include=["object", "bool"]).columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))

    feature_names = X.columns.tolist()

    # Stratified train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # Fit scaler on training data only to prevent data leakage
    scaler = StandardScaler()
    X_train = pd.DataFrame(
        scaler.fit_transform(X_train.fillna(X_train.median())),
        columns=feature_names, index=X_train.index
    )
    X_test = pd.DataFrame(
        scaler.transform(X_test.fillna(X_test.median())),
        columns=feature_names, index=X_test.index
    )

    print(f"Features: {len(feature_names)}")
    print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
    print(f"Classes: {sorted(y.unique())}")

    return X_train, X_test, y_train, y_test, scaler, feature_names


# Test on UNR-IDD first
print("=== Preprocessing UNR-IDD ===")
X_train_unr, X_test_unr, y_train_unr, y_test_unr, scaler_unr, features_unr = preprocess_dataset(
    df_unr,
    label_col="Label",
    drop_cols=["Binary Label", "Packets Rx Dropped", "Packets Tx Dropped",
               "Packets Rx Errors", "Packets Tx Errors", "Table ID", "Max Size"]
)

=== Preprocessing UNR-IDD ===
After dedup: (37410, 34)
Dropping zero-variance columns: ['Delta Packets Rx Dropped', ' Delta Packets Tx Dropped', 'Delta Packets Rx Errors', 'Delta Packets Tx Errors', 'is_valid']
Features: 21
Train: 29928 | Test: 7482
Classes: ['Blackhole', 'Diversion', 'Normal', 'Overflow', 'PortScan', 'TCP-SYN']


/var/folders/9y/hd74tjm96v78g2ll36l_z4d00000gn/T/ipykernel_8225/3121358209.py:33: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include=["object", "bool"]).columns:


In [10]:
print("=== Preprocessing CICIDS2017 ===")

# Clean CICIDS2017 label column
df_cic["Label"] = df_cic[" Label"].str.strip()

# Remove classes with too few samples
min_samples = 50
class_counts = df_cic["Label"].value_counts()
valid_classes = class_counts[class_counts >= min_samples].index.tolist()
df_cic_filtered = df_cic[df_cic["Label"].isin(valid_classes)].copy()
print(f"Classes kept: {valid_classes}")

# Replace inf values with NaN so they get handled by median imputation
df_cic_filtered.replace([float('inf'), float('-inf')], float('nan'), inplace=True)

# Sample down — take up to 20k per class
samples = []
for cls in valid_classes:
    subset = df_cic_filtered[df_cic_filtered["Label"] == cls]
    samples.append(subset.sample(min(len(subset), 20000), random_state=42))
df_cic_sample = pd.concat(samples, ignore_index=True)

print(f"Shape after sampling: {df_cic_sample.shape}")
print(f"Class distribution:\n{df_cic_sample['Label'].value_counts()}")

X_train_cic, X_test_cic, y_train_cic, y_test_cic, scaler_cic, features_cic = preprocess_dataset(
    df_cic_sample,
    label_col="Label",
    drop_cols=[" Label"]
)

=== Preprocessing CICIDS2017 ===
Classes kept: ['BENIGN', 'DoS Hulk', 'PortScan', 'DDoS', 'DoS GoldenEye', 'FTP-Patator', 'SSH-Patator', 'DoS slowloris', 'DoS Slowhttptest', 'Bot', 'Web Attack � Brute Force', 'Web Attack � XSS']
Shape after sampling: (119548, 80)
Class distribution:
Label
BENIGN                      20000
DoS Hulk                    20000
PortScan                    20000
DDoS                        20000
DoS GoldenEye               10293
FTP-Patator                  7938
SSH-Patator                  5897
DoS slowloris                5796
DoS Slowhttptest             5499
Bot                          1966
Web Attack � Brute Force     1507
Web Attack � XSS              652
Name: count, dtype: int64
After dedup: (107138, 80)
Dropping zero-variance columns: [' Bwd PSH Flags', ' Bwd URG Flags', 'Fwd Avg Bytes/Bulk', ' Fwd Avg Packets/Bulk', ' Fwd Avg Bulk Rate', ' Bwd Avg Bytes/Bulk', ' Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Features: 70
Train: 85710 | Test: 21428
Cla

In [11]:
print("=== Preprocessing UNSW-NB15 ===")

# Sample down — take up to 20k per class
unsw_classes = df_unsw["attack_cat"].value_counts()
print(f"Classes available:\n{unsw_classes}")

samples = []
for cls in unsw_classes.index:
    subset = df_unsw[df_unsw["attack_cat"] == cls]
    samples.append(subset.sample(min(len(subset), 20000), random_state=42))
df_unsw_sample = pd.concat(samples, ignore_index=True)

print(f"\nShape after sampling: {df_unsw_sample.shape}")
print(f"Class distribution:\n{df_unsw_sample['attack_cat'].value_counts()}")

X_train_unsw, X_test_unsw, y_train_unsw, y_test_unsw, scaler_unsw, features_unsw = preprocess_dataset(
    df_unsw_sample,
    label_col="attack_cat",
    drop_cols=["label", "srcip", "dstip"]
)

=== Preprocessing UNSW-NB15 ===
Classes available:
attack_cat
Normal            2218764
Generic            215481
Exploits            44525
Fuzzer              24246
DoS                 16353
Reconnaissance      13987
Analysis             2677
Backdoor             2329
Shellcode            1511
Worms                 174
Name: count, dtype: int64

Shape after sampling: (117031, 49)
Class distribution:
attack_cat
Normal            20000
Generic           20000
Exploits          20000
Fuzzer            20000
DoS               16353
Reconnaissance    13987
Analysis           2677
Backdoor           2329
Shellcode          1511
Worms               174
Name: count, dtype: int64
After dedup: (88700, 49)
Dropping high-missing columns: ['ct_flw_http_mthd', 'is_ftp_login']
Features: 43
Train: 70960 | Test: 17740
Classes: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzer', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']


/var/folders/9y/hd74tjm96v78g2ll36l_z4d00000gn/T/ipykernel_8225/3121358209.py:33: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include=["object", "bool"]).columns:


In [12]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Extended classifier configs including XGBoost and LightGBM
EXTENDED_CONFIGS = {
    **CLASSIFIER_CONFIGS,
    "XGB": {
        "class": XGBClassifier,
        "param_grid": {
            "n_estimators": {"type": "int", "low": 50, "high": 300},
            "max_depth": {"type": "int", "low": 3, "high": 10},
            "learning_rate": {"type": "float", "low": 0.01, "high": 0.3},
        },
    },
    "LGBM": {
        "class": LGBMClassifier,
        "param_grid": {
            "n_estimators": {"type": "int", "low": 50, "high": 300},
            "max_depth": {"type": "int", "low": 3, "high": 10},
            "learning_rate": {"type": "float", "low": 0.01, "high": 0.3},
            "num_leaves": {"type": "int", "low": 20, "high": 100},
        },
    },
}

print(f"Total classifiers: {list(EXTENDED_CONFIGS.keys())}")

Total classifiers: ['KNN', 'LR', 'RF', 'DT', 'AB', 'XGB', 'LGBM']


In [13]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

print("--- LGBM on UNSW-NB15 ---")
model = LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    num_leaves=63,
    verbose=-1
)
model.fit(X_train_unsw, y_train_unsw)
y_pred = model.predict(X_test_unsw)
acc = accuracy_score(y_test_unsw, y_pred)
print(f"Test accuracy: {acc:.4f}")
print(classification_report(y_test_unsw, y_pred))
joblib.dump(model, "../results/LGBM_UNSW-NB15_model.pkl")
print("Model saved.")

--- LGBM on UNSW-NB15 ---
Test accuracy: 0.7722
                precision    recall  f1-score   support

      Analysis       0.15      0.22      0.18       437
      Backdoor       0.14      0.14      0.14       397
           DoS       0.28      0.34      0.31      1133
      Exploits       0.68      0.63      0.66      2796
        Fuzzer       0.87      0.80      0.83      3643
       Generic       0.97      0.96      0.96      2353
        Normal       0.96      0.96      0.96      3974
Reconnaissance       0.81      0.81      0.81      2671
     Shellcode       0.74      0.81      0.77       302
         Worms       0.05      0.06      0.05        34

      accuracy                           0.77     17740
     macro avg       0.56      0.57      0.57     17740
  weighted avg       0.79      0.77      0.78     17740

Model saved.


In [14]:
import time
import joblib
import numpy as np
from sklearn.metrics import accuracy_score

datasets = {
    "UNR-IDD": (X_test_unr, y_test_unr),
    "CICIDS2017": (X_test_cic, y_test_cic),
    "UNSW-NB15": (X_test_unsw, y_test_unsw)
}

model_names = ["KNN", "LR", "RF", "DT", "AB", "XGB", "LGBM"]

for dataset_name, (X_test, y_test) in datasets.items():
    print(f"\n=== {dataset_name} ===")
    for name in model_names:
        try:
            model = joblib.load(f"../results/{name}_{dataset_name}_model.pkl")
            start = time.time()
            y_pred = model.predict(X_test)
            elapsed = (time.time() - start) * 1000
            acc = accuracy_score(y_test, y_pred)
            print(f"{name}: {acc:.4f} accuracy | {elapsed:.2f} ms inference time")
        except Exception as e:
            print(f"{name}: ERROR - {e}")


=== UNR-IDD ===
KNN: 0.8786 accuracy | 98.86 ms inference time
LR: 0.8344 accuracy | 0.72 ms inference time
RF: 0.9508 accuracy | 127.55 ms inference time
DT: 0.9393 accuracy | 0.90 ms inference time
AB: 0.7968 accuracy | 133.15 ms inference time
XGB: 0.0000 accuracy | 14.15 ms inference time
LGBM: 0.9722 accuracy | 65.87 ms inference time

=== CICIDS2017 ===
KNN: 0.9843 accuracy | 884.22 ms inference time
LR: 0.9706 accuracy | 1.59 ms inference time
RF: 0.9923 accuracy | 219.58 ms inference time
DT: 0.9912 accuracy | 1.67 ms inference time
AB: 0.7970 accuracy | 513.98 ms inference time
XGB: 0.0000 accuracy | 68.12 ms inference time
LGBM: 0.9922 accuracy | 97.09 ms inference time

=== UNSW-NB15 ===
KNN: 0.7503 accuracy | 455.53 ms inference time
LR: 0.7657 accuracy | 0.99 ms inference time
RF: 0.8175 accuracy | 56.93 ms inference time
DT: 0.8228 accuracy | 1.36 ms inference time
AB: 0.7657 accuracy | 454.09 ms inference time
XGB: 0.0000 accuracy | 58.01 ms inference time
LGBM: 0.7722 

In [15]:
from sklearn.preprocessing import LabelEncoder
import time

le_unr = LabelEncoder()
le_cic = LabelEncoder()
le_unsw = LabelEncoder()

le_unr.fit(y_train_unr)
le_cic.fit(y_train_cic)
le_unsw.fit(y_train_unsw)

xgb_datasets = {
    "UNR-IDD": (X_test_unr, y_test_unr, le_unr),
    "CICIDS2017": (X_test_cic, y_test_cic, le_cic),
    "UNSW-NB15": (X_test_unsw, y_test_unsw, le_unsw)
}

for dataset_name, (X_test, y_test, le) in xgb_datasets.items():
    model = joblib.load(f"../results/XGB_{dataset_name}_model.pkl")
    start = time.time()
    y_pred_enc = model.predict(X_test)
    elapsed = (time.time() - start) * 1000
    y_pred = le.inverse_transform(y_pred_enc)
    acc = accuracy_score(y_test, y_pred)
    print(f"XGB {dataset_name}: {acc:.4f} accuracy | {elapsed:.2f} ms inference time")

XGB UNR-IDD: 0.9648 accuracy | 14.81 ms inference time
XGB CICIDS2017: 0.9926 accuracy | 67.15 ms inference time
XGB UNSW-NB15: 0.8293 accuracy | 55.18 ms inference time
